# IT8532 Assessment 2: Fine-Tuning Qwen2.5-3B-Instruct with Unsloth QLoRA on Google Colab
**Task:** Fine-tuning `Qwen2.5-3B-Instruct` for natural language task parameter extraction into strict JSON schema format.
**Google Drive Integration:** Mounts Google Drive to load `finetune_dataset.json` and save the trained GGUF model (`Q4_K_M`).
**Target Deployment:** Proxmox CT 200 (`inference-llm`, IP: `192.168.10.20`) via `llama.cpp` CPU GGUF (`Q4_K_M`).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
import os

# Mount Google Drive to /content/drive
drive.mount('/content/drive')

# Define persistent project directory in your Google Drive
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Assessment_2"
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print(f"Google Drive mounted successfully. Project directory: {DRIVE_PROJECT_DIR}")

## 2. Environment Setup & Dependency Installation

In [ ]:
# Install Unsloth and required fine-tuning dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets trl sentencepiece
!pip install gguf

## 3. Load Base Model & Tokenizer (Qwen2.5-3B-Instruct in 4-bit)

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None  # None for auto detection (Float16 for Tesla T4, Bfloat16 for Ampere+)
load_in_4bit = True  # Use 4bit quantization to reduce VRAM usage

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

## 4. Configure QLoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,            # LoRA Rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,   # LoRA Scaling Factor
    lora_dropout=0,  # Optimized at 0 for Unsloth
    bias="none",     # Optimized at none for Unsloth
    use_gradient_checkpointing="unsloth", # 30% less VRAM consumption
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

print("QLoRA Adapters initialized successfully.")

## 5. Load & Format Dataset from Google Drive (`finetune_dataset.json`)

In [ ]:
import json
from google.colab import files
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Set up Qwen-2.5 ChatML chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
    mapping={"role": "role", "content": "content", "user": "user", "assistant": "assistant"}
)

DATASET_PATH = os.path.join(DRIVE_PROJECT_DIR, "finetune_dataset.json")

# Check if dataset is already in Google Drive; if not, upload it
if not os.path.exists(DATASET_PATH):
    print(f"Dataset file not found at {DATASET_PATH}.")
    print("Please select and upload finetune_dataset.json from your local PC:")
    uploaded = files.upload()
    for fname in uploaded.keys():
        target_file = os.path.join(DRIVE_PROJECT_DIR, fname)
        with open(target_file, "wb") as f:
            f.write(uploaded[fname])
        print(f"Saved {fname} to Google Drive: {target_file}")

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

def formatting_prompts_func(examples):
    convs = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convs]
    return {"text": texts}

dataset = Dataset.from_dict({"conversations": [item["conversations"] for item in raw_data]})
dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"Successfully loaded dataset from Google Drive: {len(dataset)} instruction examples.")
print("Sample formatted prompt:\n", dataset[0]["text"][:300])

## 6. Train Model with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False, # Faster training for short sequences
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=150, # Or use num_train_epochs=3
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
    ),
)

# Start Fine-Tuning
trainer_stats = trainer.train()
print("Qwen2.5-3B Fine-Tuning Complete!")

## 7. Evaluation & Inference Verification

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are a task extraction assistant for a time management system. Extract scheduling information from the user's natural language task description and return ONLY a valid JSON object with exactly these fields: task, deadline_hours, duration_hours, urgency, importance, has_dependency, dependency_note."},
    {"role": "user", "content": "Finish the critical security audit for the payment system by tomorrow afternoon. Takes about 3 hours. Blocked until the API integration is complete."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=256, use_cache=True)
response = tokenizer.batch_decode(outputs)
print("Generated Response:\n", response[0].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", ""))

## 8. Export & Save GGUF (`Q4_K_M`) Directly to Google Drive

In [ ]:
# Save merged 4-bit GGUF model directly to Google Drive
GGUF_SAVE_DIR = os.path.join(DRIVE_PROJECT_DIR, "qwen2.5-3b-task-assistant-gguf")
model.save_pretrained_gguf(GGUF_SAVE_DIR, tokenizer, quantization_method="q4_k_m")

print(f"GGUF Export Complete! Saved directly to Google Drive:")
print(f" 👉 {GGUF_SAVE_DIR}/qwen2.5-3b-task-assistant-Q4_K_M.gguf")

# Also trigger browser download
gguf_file = os.path.join(GGUF_SAVE_DIR, "qwen2.5-3b-task-assistant-Q4_K_M.gguf")
if os.path.exists(gguf_file):
    files.download(gguf_file)